## quick start on autogen website
# 1. Install autogen
```bash
pip install -U "autogen-agentchat" "autogen-ext[openai,azure]"
```

In [1]:
! conda create -n autogen python=3.10

! pip install -U "autogen-agentchat" "autogen-ext[openai,azure]"



Remove existing environment?
This will remove ALL directories contained within this specified prefix directory, including any other conda environments.

 (y/[n])? 
CondaSystemExit: 
Operation aborted.  Exiting.

^C


In [3]:
! pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.1.0-py3-none-any.whl.metadata (24 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.1.0-py3-none-any.whl (20 kB)


In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
import os
from dotenv import load_dotenv
import json

load_dotenv("/etc/.env")
# Define a model client. You can use other 

model = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
key = os.getenv("AZURE_OPENAI_API_KEY")

model_client = AzureOpenAIChatCompletionClient(
    model=model,
    api_key=key,
    endpoint=endpoint,
    api_version="2025-03-01-preview",
    
    # api_key="YOUR_API_KEY",
)


# Define a simple function tool that the agent can use.
# For this example, we use a fake weather tool for demonstration purposes.
async def get_weather(city: str) -> str:
    """Get the weather for a given city."""
    # Simulate a weather API response.
    # 比如说Azure Maps API
    json_str = f'''
    {{
        "city": "{city}",
        "weather": {{
            "temperature": 73,
            "condition": "Sunny"
        }}
    }}
    '''
    json_str = json_str.replace("\n", "")
    data = json.loads(json_str)
    city = data["city"]
    temperature = data["weather"]["temperature"]
    condition = data["weather"]["condition"]

    result = f"The weather in {city} is {temperature}°F and {condition}."

    
    # print(f"get_weather({city}) called")
    print(f"The result is: {result}")
    # 返回天气数据信息
    return result


# Define an AssistantAgent with the model, tool, system message, and reflection enabled.
# The system message instructs the agent via natural language.
agent = AssistantAgent(
    name="weather_agent",
    model_client=model_client,
    tools=[get_weather],
    system_message="You are a helpful assistant.",
    reflect_on_tool_use=True,
    model_client_stream=True,  # Enable streaming tokens from the model client.
)


# Run the agent and stream the messages to the console.
async def main() -> None:
    await Console(agent.run_stream(task="What is the weather in Shanghai, answer it in Chinese?"))
    # Close the connection to the model client.
    await model_client.close()


# NOTE: if running this inside a Python script you'll need to use asyncio.run(main()).
await main()


---------- TextMessage (user) ----------
What is the weather in Shanghai, answer it in Chinese?


---------- ToolCallRequestEvent (weather_agent) ----------
[FunctionCall(id='call_HuR54Jwwz7UsgqDrHBBhRwyU', arguments='{"city":"Shanghai"}', name='get_weather')]
The result is: The weather in Shanghai is 73°F and Sunny.
---------- ToolCallExecutionEvent (weather_agent) ----------
[FunctionExecutionResult(content='The weather in Shanghai is 73°F and Sunny.', name='get_weather', call_id='call_HuR54Jwwz7UsgqDrHBBhRwyU', is_error=False)]
[FunctionCall(id='call_HuR54Jwwz7UsgqDrHBBhRwyU', arguments='{"city":"Shanghai"}', name='get_weather')]
The result is: The weather in Shanghai is 73°F and Sunny.
---------- ToolCallExecutionEvent (weather_agent) ----------
[FunctionExecutionResult(content='The weather in Shanghai is 73°F and Sunny.', name='get_weather', call_id='call_HuR54Jwwz7UsgqDrHBBhRwyU', is_error=False)]
---------- ModelClientStreamingChunkEvent (weather_agent) ----------
上海的天气是晴朗，气温为73°F（约23°C）。
